## Copyright 2022 Google LLC. Double-click for license information.

In [1]:
# Copyright 2022 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Null-text inversion + Editing with Prompt-to-Prompt

In [1]:
from typing import Optional, Union, Tuple, List, Callable, Dict
from tqdm.notebook import tqdm
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler
import torch.nn.functional as nnf
import numpy as np
import abc
import ptp_utils
import seq_aligner
import shutil
from torch.optim.adam import Adam
from PIL import Image

For loading the Stable Diffusion using Diffusers, follow the instuctions https://huggingface.co/blog/stable_diffusion and update MY_TOKEN with your token.

In [2]:
# MY_TOKEN = ''
LOW_RESOURCE = False 
NUM_DIFFUSION_STEPS = 10
GUIDANCE_SCALE = 3
MAX_NUM_WORDS = 77 * 4
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

scheduler = DDIMScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", clip_sample=False, set_alpha_to_one=False)
ldm_stable = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4", local_files_only=True, scheduler=scheduler).to(device)
try:
    ldm_stable.disable_xformers_memory_efficient_attention()
except AttributeError:
    print("Attribute disable_xformers_memory_efficient_attention() is missing")
tokenizer = ldm_stable.tokenizer

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/djm/ERDDCI/.venv/lib/python3.11/site-packages/diffusers/pipelines/stable_diffusion/pipeline_stable_diffusion.py:223: FutureWarning: The configuration file of this scheduler: DDIMScheduler {
  "_class_name": "DDIMScheduler",
  "_diffusers_version": "0.33.1",
  "beta_end": 0.012,
  "beta_schedule": "scaled_linear",
  "beta_start": 0.00085,
  "clip_sample": false,
  "clip_sample_range": 1.0,
  "dynamic_thresholding_ratio": 0.995,
  "num_train_timesteps": 1000,
  "prediction_type": "epsilon",
  "rescale_betas_zero_snr": false,
  "sample_max_value": 1.0,
  "set_alpha_to_one": false,
  "steps_offset": 0,
  "thresholding": false,
  "timestep_spacing": "leading",
  "trained_betas": null
}
 is outdated. `steps_offset` should be set to 1 instead of 0. Please make sure to update the config accordingly as leaving `steps_offset` might led to incorrect results in future versions. If you have downloaded this checkpoint from the Hugging Face Hub, it would be very nice if you could open a Pull re

## Prompt-to-Prompt code

In [3]:
class LocalBlend:
    
    def get_mask(self, maps, alpha, use_pool):
        k = 1
        maps = (maps * alpha).sum(-1).mean(1)
        if use_pool:
            maps = nnf.max_pool2d(maps, (k * 2 + 1, k * 2 +1), (1, 1), padding=(k, k))
        mask = nnf.interpolate(maps, size=(x_t.shape[2:]))
        mask = mask / mask.max(2, keepdims=True)[0].max(3, keepdims=True)[0]
        mask = mask.gt(self.th[1-int(use_pool)])
        mask = mask[:1] + mask
        return mask
    
    def __call__(self, x_t, attention_store):
        self.counter += 1
        if self.counter > self.start_blend:
            maps = attention_store["down_cross"][2:4] + attention_store["up_cross"][:3]
            maps = [item.reshape(self.alpha_layers.shape[0], -1, 1, 16, 16, MAX_NUM_WORDS) for item in maps]
            maps = torch.cat(maps, dim=1)
            mask = self.get_mask(maps, self.alpha_layers, True)
            if self.substruct_layers is not None:
                maps_sub = ~self.get_mask(maps, self.substruct_layers, False)
                mask = mask * maps_sub
            mask = mask.float()
            x_t = x_t[:1] + mask * (x_t - x_t[:1])
        return x_t
       
    def __init__(self, prompts: List[str], words: [List[List[str]]], substruct_words=None, start_blend=0.2, th=(.3, .3)):
        alpha_layers = torch.zeros(len(prompts),  1, 1, 1, 1, MAX_NUM_WORDS)
        for i, (prompt, words_) in enumerate(zip(prompts, words)):
            if type(words_) is str:
                words_ = [words_]
            for word in words_:
                ind = ptp_utils.get_word_inds(prompt, word, tokenizer)
                alpha_layers[i, :, :, :, :, ind] = 1
        
        if substruct_words is not None:
            substruct_layers = torch.zeros(len(prompts),  1, 1, 1, 1, MAX_NUM_WORDS)
            for i, (prompt, words_) in enumerate(zip(prompts, substruct_words)):
                if type(words_) is str:
                    words_ = [words_]
                for word in words_:
                    ind = ptp_utils.get_word_inds(prompt, word, tokenizer)
                    substruct_layers[i, :, :, :, :, ind] = 1
            self.substruct_layers = substruct_layers.to(device)
        else:
            self.substruct_layers = None
        self.alpha_layers = alpha_layers.to(device)
        self.start_blend = int(start_blend * NUM_DIFFUSION_STEPS)
        self.counter = 0 
        self.th=th


        
        
class EmptyControl:
    
    
    def step_callback(self, x_t):
        return x_t
    
    def between_steps(self):
        return
    
    def __call__(self, attn, is_cross: bool, place_in_unet: str):
        return attn

    
class AttentionControl(abc.ABC):
    
    def step_callback(self, x_t):
        return x_t
    
    def between_steps(self):
        return
    
    @property
    def num_uncond_att_layers(self):
        return self.num_att_layers if LOW_RESOURCE else 0
    
    @abc.abstractmethod
    def forward (self, attn, is_cross: bool, place_in_unet: str):
        raise NotImplementedError

    def __call__(self, attn, is_cross: bool, place_in_unet: str):
        if self.cur_att_layer >= self.num_uncond_att_layers:
            if LOW_RESOURCE:
                attn = self.forward(attn, is_cross, place_in_unet)
            else:
                h = attn.shape[0]
                attn[h // 2:] = self.forward(attn[h // 2:], is_cross, place_in_unet)
        self.cur_att_layer += 1
        if self.cur_att_layer == self.num_att_layers + self.num_uncond_att_layers:
            self.cur_att_layer = 0
            self.cur_step += 1
            self.between_steps()
        return attn
    
    def reset(self):
        self.cur_step = 0
        self.cur_att_layer = 0

    def __init__(self):
        self.cur_step = 0
        self.num_att_layers = -1
        self.cur_att_layer = 0

class SpatialReplace(EmptyControl):
    
    def step_callback(self, x_t):
        if self.cur_step < self.stop_inject:
            b = x_t.shape[0]
            x_t = x_t[:1].expand(b, *x_t.shape[1:])
        return x_t

    def __init__(self, stop_inject: float):
        super(SpatialReplace, self).__init__()
        self.stop_inject = int((1 - stop_inject) * NUM_DIFFUSION_STEPS)
        

class AttentionStore(AttentionControl):

    @staticmethod
    def get_empty_store():
        return {"down_cross": [], "mid_cross": [], "up_cross": [],
                "down_self": [],  "mid_self": [],  "up_self": []}

    def forward(self, attn, is_cross: bool, place_in_unet: str):
        key = f"{place_in_unet}_{'cross' if is_cross else 'self'}"
        if attn.shape[1] <= 32 ** 2:  # avoid memory overhead
            self.step_store[key].append(attn)
        return attn

    def between_steps(self):
        if len(self.attention_store) == 0:
            self.attention_store = self.step_store
        else:
            for key in self.attention_store:
                for i in range(len(self.attention_store[key])):
                    self.attention_store[key][i] += self.step_store[key][i]
        self.step_store = self.get_empty_store()

    def get_average_attention(self):
        average_attention = {key: [item / self.cur_step for item in self.attention_store[key]] for key in self.attention_store}
        return average_attention


    def reset(self):
        super(AttentionStore, self).reset()
        self.step_store = self.get_empty_store()
        self.attention_store = {}

    def __init__(self):
        super(AttentionStore, self).__init__()
        self.step_store = self.get_empty_store()
        self.attention_store = {}

        
class AttentionControlEdit(AttentionStore, abc.ABC):
    
    def step_callback(self, x_t):
        if self.local_blend is not None:
            x_t = self.local_blend(x_t, self.attention_store)
        return x_t
        
    def replace_self_attention(self, attn_base, att_replace, place_in_unet):
        if att_replace.shape[2] <= 32 ** 2:
            attn_base = attn_base.unsqueeze(0).expand(att_replace.shape[0], *attn_base.shape)
            return attn_base
        else:
            return att_replace
    
    @abc.abstractmethod
    def replace_cross_attention(self, attn_base, att_replace):
        raise NotImplementedError
    
    def forward(self, attn, is_cross: bool, place_in_unet: str):
        super(AttentionControlEdit, self).forward(attn, is_cross, place_in_unet)
        if is_cross or (self.num_self_replace[0] <= self.cur_step < self.num_self_replace[1]):
            h = attn.shape[0] // (self.batch_size)
            attn = attn.reshape(self.batch_size, h, *attn.shape[1:])
            attn_base, attn_repalce = attn[0], attn[1:]
            if is_cross:
                alpha_words = self.cross_replace_alpha[self.cur_step]
                attn_repalce_new = self.replace_cross_attention(attn_base, attn_repalce) * alpha_words + (1 - alpha_words) * attn_repalce
                attn[1:] = attn_repalce_new
            else:
                attn[1:] = self.replace_self_attention(attn_base, attn_repalce, place_in_unet)
            attn = attn.reshape(self.batch_size * h, *attn.shape[2:])
        return attn
    
    def __init__(self, prompts, num_steps: int,
                 cross_replace_steps: Union[float, Tuple[float, float], Dict[str, Tuple[float, float]]],
                 self_replace_steps: Union[float, Tuple[float, float]],
                 local_blend: Optional[LocalBlend]):
        super(AttentionControlEdit, self).__init__()
        self.batch_size = len(prompts)
        self.cross_replace_alpha = ptp_utils.get_time_words_attention_alpha(prompts, num_steps, cross_replace_steps, tokenizer).to(device)
        if type(self_replace_steps) is float:
            self_replace_steps = 0, self_replace_steps
        self.num_self_replace = int(num_steps * self_replace_steps[0]), int(num_steps * self_replace_steps[1])
        self.local_blend = local_blend

class AttentionReplace(AttentionControlEdit):

    def replace_cross_attention(self, attn_base, att_replace):
        return torch.einsum('hpw,bwn->bhpn', attn_base, self.mapper)
      
    def __init__(self, prompts, num_steps: int, cross_replace_steps: float, self_replace_steps: float,
                 local_blend: Optional[LocalBlend] = None):
        super(AttentionReplace, self).__init__(prompts, num_steps, cross_replace_steps, self_replace_steps, local_blend)
        self.mapper = seq_aligner.get_replacement_mapper(prompts, tokenizer).to(device)
        

class AttentionRefine(AttentionControlEdit):

    def replace_cross_attention(self, attn_base, att_replace):
        attn_base_replace = attn_base[:, :, self.mapper].permute(2, 0, 1, 3)
        attn_replace = attn_base_replace * self.alphas + att_replace * (1 - self.alphas)
        # attn_replace = attn_replace / attn_replace.sum(-1, keepdims=True)
        return attn_replace

    def __init__(self, prompts, num_steps: int, cross_replace_steps: float, self_replace_steps: float,
                 local_blend: Optional[LocalBlend] = None):
        super(AttentionRefine, self).__init__(prompts, num_steps, cross_replace_steps, self_replace_steps, local_blend)
        self.mapper, alphas = seq_aligner.get_refinement_mapper(prompts, tokenizer)
        self.mapper, alphas = self.mapper.to(device), alphas.to(device)
        self.alphas = alphas.reshape(alphas.shape[0], 1, 1, alphas.shape[1])


class AttentionReweight(AttentionControlEdit):

    def replace_cross_attention(self, attn_base, att_replace):
        if self.prev_controller is not None:
            attn_base = self.prev_controller.replace_cross_attention(attn_base, att_replace)
        attn_replace = attn_base[None, :, :, :] * self.equalizer[:, None, None, :]
        # attn_replace = attn_replace / attn_replace.sum(-1, keepdims=True)
        return attn_replace

    def __init__(self, prompts, num_steps: int, cross_replace_steps: float, self_replace_steps: float, equalizer,
                local_blend: Optional[LocalBlend] = None, controller: Optional[AttentionControlEdit] = None):
        super(AttentionReweight, self).__init__(prompts, num_steps, cross_replace_steps, self_replace_steps, local_blend)
        self.equalizer = equalizer.to(device)
        self.prev_controller = controller


def get_equalizer(text: str, word_select: Union[int, Tuple[int, ...]], values: Union[List[float],
                  Tuple[float, ...]]):
    if type(word_select) is int or type(word_select) is str:
        word_select = (word_select,)
    equalizer = torch.ones(1, 77)
    
    for word, val in zip(word_select, values):
        inds = ptp_utils.get_word_inds(text, word, tokenizer)
        equalizer[:, inds] = val
    return equalizer

def aggregate_attention(attention_store: AttentionStore, res: int, from_where: List[str], is_cross: bool, select: int):
    out = []
    attention_maps = attention_store.get_average_attention()
    num_pixels = res ** 2
    for location in from_where:
        for item in attention_maps[f"{location}_{'cross' if is_cross else 'self'}"]:
            if item.shape[1] == num_pixels:
                cross_maps = item.reshape(len(prompts), -1, res, res, item.shape[-1])[select]
                out.append(cross_maps)
    out = torch.cat(out, dim=0)
    out = out.sum(0) / out.shape[0]
    return out.cpu()


def make_controller(prompts: List[str], is_replace_controller: bool, cross_replace_steps: Dict[str, float], self_replace_steps: float, blend_words=None, equilizer_params=None) -> AttentionControlEdit:
    if blend_words is None:
        lb = None
    else:
        lb = LocalBlend(prompts, blend_word)
    if is_replace_controller:
        controller = AttentionReplace(prompts, NUM_DIFFUSION_STEPS, cross_replace_steps=cross_replace_steps, self_replace_steps=self_replace_steps, local_blend=lb)
    else:
        controller = AttentionRefine(prompts, NUM_DIFFUSION_STEPS, cross_replace_steps=cross_replace_steps, self_replace_steps=self_replace_steps, local_blend=lb)
    if equilizer_params is not None:
        eq = get_equalizer(prompts[1], equilizer_params["words"], equilizer_params["values"])
        controller = AttentionReweight(prompts, NUM_DIFFUSION_STEPS, cross_replace_steps=cross_replace_steps,
                                       self_replace_steps=self_replace_steps, equalizer=eq, local_blend=lb, controller=controller)
    return controller


def show_cross_attention(attention_store: AttentionStore, res: int, from_where: List[str], select: int = 0):
    tokens = tokenizer.encode(prompts[select])
    decoder = tokenizer.decode
    attention_maps = aggregate_attention(attention_store, res, from_where, True, select)
    images = []
    for i in range(len(tokens)):
        image = attention_maps[:, :, i]
        image = 255 * image / image.max()
        image = image.unsqueeze(-1).expand(*image.shape, 3)
        image = image.numpy().astype(np.uint8)
        image = np.array(Image.fromarray(image).resize((256, 256)))
        image = ptp_utils.text_under_image(image, decoder(int(tokens[i])))
        images.append(image)
    ptp_utils.view_images(np.stack(images, axis=0))
    

def show_self_attention_comp(attention_store: AttentionStore, res: int, from_where: List[str],
                        max_com=10, select: int = 0):
    attention_maps = aggregate_attention(attention_store, res, from_where, False, select).numpy().reshape((res ** 2, res ** 2))
    u, s, vh = np.linalg.svd(attention_maps - np.mean(attention_maps, axis=1, keepdims=True))
    images = []
    for i in range(max_com):
        image = vh[i].reshape(res, res)
        image = image - image.min()
        image = 255 * image / image.max()
        image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2).astype(np.uint8)
        image = Image.fromarray(image).resize((256, 256))
        image = np.array(image)
        images.append(image)
    ptp_utils.view_images(np.concatenate(images, axis=1))

In [4]:
def load_512(image_or_path):
    """input PIL Image or image path, resize to 512x512"""
    if isinstance(image_or_path, str):
        image = np.array(Image.open(image_or_path))[:, :, :3]
    elif isinstance(image_or_path, Image.Image):
        image = np.array(image_or_path.convert("RGB"))
    else:
        raise TypeError
    image = np.array(Image.fromarray(image).resize((512, 512)))
    return image

## Null Text Inversion code

In [5]:
class NullInversion:
    def __init__(self, model):
        self.model = model
        self.tokenizer = self.model.tokenizer
        self.scheduler = self.model.scheduler
        self.model.scheduler.set_timesteps(NUM_DIFFUSION_STEPS)
        self.prompt = None
        self.context = None

    @torch.no_grad()
    def latent2image(self, latents, return_type='np'):
        latents = 1 / 0.18215 * latents.detach()
        image = self.model.vae.decode(latents)['sample']
        if return_type == 'np':
            image = (image / 2 + 0.5).clamp(0, 1)
            image = image.cpu().permute(0, 2, 3, 1).numpy()[0]
            image = (image * 255).astype(np.uint8)
        return image

    @torch.no_grad()
    def image2latent(self, image):
        with torch.no_grad():
            if type(image) is Image:
                image = np.array(image)
            if type(image) is torch.Tensor and image.dim() == 4:
                latents = image
            else:
                image = torch.from_numpy(image).float() / 127.5 - 1
                image = image.permute(2, 0, 1).unsqueeze(0).to(device)
                latents = self.model.vae.encode(image)['latent_dist'].mean
                latents = latents * 0.18215
        return latents
    
    @torch.no_grad()
    def init_prompt(self, prompt: str):
        uncond_input = self.model.tokenizer(
            [""], padding="max_length", max_length=self.model.tokenizer.model_max_length,
            return_tensors="pt"
        )
        uncond_embeddings = self.model.text_encoder(uncond_input.input_ids.to(self.model.device))[0]
        text_input = self.model.tokenizer(
            [prompt],
            padding="max_length",
            max_length=self.model.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )
        text_embeddings = self.model.text_encoder(text_input.input_ids.to(self.model.device))[0]
        self.context = torch.cat([uncond_embeddings, text_embeddings])
        self.prompt = prompt

    def prev_step(self, model_output: Union[torch.FloatTensor, np.ndarray], timestep: int, sample: Union[torch.FloatTensor, np.ndarray]):
        prev_timestep = timestep - self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        alpha_prod_t = self.scheduler.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.scheduler.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else self.scheduler.final_alpha_cumprod
        beta_prod_t = 1 - alpha_prod_t
        pred_original_sample = (sample - beta_prod_t ** 0.5 * model_output) / alpha_prod_t ** 0.5
        pred_sample_direction = (1 - alpha_prod_t_prev) ** 0.5 * model_output
        prev_sample = alpha_prod_t_prev ** 0.5 * pred_original_sample + pred_sample_direction
        return prev_sample
    
    def next_step(self, model_output: Union[torch.FloatTensor, np.ndarray], timestep: int, sample: Union[torch.FloatTensor, np.ndarray]):
        timestep, next_timestep = min(timestep - self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps, 999), timestep
        alpha_prod_t = self.scheduler.alphas_cumprod[timestep] if timestep >= 0 else self.scheduler.final_alpha_cumprod
        alpha_prod_t_next = self.scheduler.alphas_cumprod[next_timestep]
        beta_prod_t = 1 - alpha_prod_t
        next_original_sample = (sample - beta_prod_t ** 0.5 * model_output) / alpha_prod_t ** 0.5
        next_sample_direction = (1 - alpha_prod_t_next) ** 0.5 * model_output
        next_sample = alpha_prod_t_next ** 0.5 * next_original_sample + next_sample_direction
        return next_sample
    
    def get_noise_pred_single(self, latents, t, context):
        noise_pred = self.model.unet(latents, t, encoder_hidden_states=context)["sample"]
        return noise_pred

    def get_noise_pred(self, latents, t, guidance_scale=1.0, context=None):
        latents_input = torch.cat([latents] * 2)
        if context is None:
            context = self.context
        noise_pred = self.model.unet(latents_input, t, encoder_hidden_states=context)["sample"]
        noise_pred_uncond, noise_prediction_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_prediction_text - noise_pred_uncond)

        return noise_pred


    @torch.no_grad()
    def ddim_loop(self, latent):
        uncond_embeddings, cond_embeddings = self.context.chunk(2)
        all_latent = [latent]
        latent = latent.clone().detach()
        for i in range(NUM_DIFFUSION_STEPS):
            t = self.model.scheduler.timesteps[len(self.model.scheduler.timesteps) - i - 1]
            noise_pred = self.get_noise_pred_single(latent, t, cond_embeddings)
            latent = self.next_step(noise_pred, t, latent)
            all_latent.append(latent)
        return all_latent # [0, .., t, t+1， ..., T]

    @torch.no_grad()
    def ddim_inversion(self, image):
        latent = self.image2latent(image)
        image_rec = self.latent2image(latent)
        ddim_latents = self.ddim_loop(latent)
        return image_rec, ddim_latents

    def null_optimization(self, latents, num_inner_steps, epsilon):
        uncond_embeddings, cond_embeddings = self.context.chunk(2)
        uncond_embeddings_list = []
        latent_cur = latents[-1] # z_t(init as z_T)
        bar = tqdm(total=num_inner_steps * NUM_DIFFUSION_STEPS)
        for i in range(NUM_DIFFUSION_STEPS):
            uncond_embeddings = uncond_embeddings.clone().detach()
            uncond_embeddings.requires_grad = True
            optimizer = Adam([uncond_embeddings], lr=1e-2 * (1. - i / 100.))
            latent_prev = latents[len(latents) - i - 2] # z_(t-1)
            t = self.model.scheduler.timesteps[i]
            with torch.no_grad():
                noise_pred_cond = self.get_noise_pred_single(latent_cur, t, cond_embeddings)
            for j in range(num_inner_steps):
                noise_pred_uncond = self.get_noise_pred_single(latent_cur, t, uncond_embeddings)
                noise_pred = noise_pred_uncond + GUIDANCE_SCALE * (noise_pred_cond - noise_pred_uncond)
                latents_prev_rec = self.prev_step(noise_pred, t, latent_cur) # z_(t-1)^recon
                loss = nnf.mse_loss(latents_prev_rec, latent_prev)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                loss_item = loss.item()
                bar.update()
                if loss_item < epsilon + i * 2e-5:
                    break
            for j in range(j + 1, num_inner_steps):
                bar.update()
            uncond_embeddings_list.append(uncond_embeddings[:1].detach())
            with torch.no_grad():
                context = torch.cat([uncond_embeddings, cond_embeddings])
                noise_pred = self.get_noise_pred(latent_cur, t, GUIDANCE_SCALE, context)
                latent_cur = self.prev_step(noise_pred, t, latent_cur) # z_(t-1)^(optim)
        bar.close()
        return uncond_embeddings_list # [T, ..., t, t-1, ..., 0]
    
    def invert(self, image_path: str, prompt: str, offsets=(0,0,0,0), num_inner_steps=10, early_stop_epsilon=1e-5, verbose=False):
        self.init_prompt(prompt)
        ptp_utils.register_attention_control(self.model, None)
        image_gt = load_512(image_path)
        if verbose:
            print("DDIM inversion...")
        image_rec, ddim_latents = self.ddim_inversion(image_gt)
        if verbose:
            print("Null-text optimization...")
        uncond_embeddings = self.null_optimization(ddim_latents, num_inner_steps, early_stop_epsilon)
        # uncond_embeddings = None
        return (image_gt, image_rec), ddim_latents[-1], uncond_embeddings
        
    def get_encode_invert_inference_latent(self, image_or_path, prompt: str, guidance_scale=GUIDANCE_SCALE, num_inner_steps=10, early_stop_epsilon=1e-5):
        with torch.no_grad():
            self.init_prompt(prompt)
            _, cond_embeddings = self.context.chunk(2)
            ptp_utils.register_attention_control(self.model, None)

            image_gt = load_512(image_or_path)
            encode_latent = self.image2latent(image_gt)

            ddim_latent_list = self.ddim_loop(encode_latent)
            ddim_invert_latent = ddim_latent_list[-1].clone().detach()

        uncond_embeddings_list = self.null_optimization(ddim_latent_list, num_inner_steps, early_stop_epsilon)
        
        with torch.no_grad():
            nti_inference_latent = ddim_invert_latent # z_t(init as z_T)
            for i in range(NUM_DIFFUSION_STEPS):
                t = self.model.scheduler.timesteps[i] # [T, ..., t, t-1, ..., 0]
                opt_uncond_embedding_t = uncond_embeddings_list[i]
                opt_context = torch.cat([opt_uncond_embedding_t.expand(*cond_embeddings.shape), cond_embeddings])
                # print(f"[Debug] inference: {i=}\t{t=}")
                noise_pred = self.get_noise_pred(nti_inference_latent, t, GUIDANCE_SCALE, context=opt_context)
                nti_inference_latent = self.prev_step(noise_pred, t, nti_inference_latent)
        
        return encode_latent, ddim_invert_latent, nti_inference_latent

null_inversion = NullInversion(ldm_stable)


## Infernce Code

In [6]:
@torch.no_grad()
def text2image_ldm_stable(
    model,
    prompt:  List[str],
    controller,
    num_inference_steps: int = 50,
    guidance_scale: Optional[float] = 7.5,
    generator: Optional[torch.Generator] = None,
    latent: Optional[torch.FloatTensor] = None,
    uncond_embeddings=None,
    start_time=50,
    return_type='image'
):
    batch_size = len(prompt)
    ptp_utils.register_attention_control(model, controller)
    height = width = 512
    
    text_input = model.tokenizer(
        prompt,
        padding="max_length",
        max_length=model.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    text_embeddings = model.text_encoder(text_input.input_ids.to(model.device))[0]
    max_length = text_input.input_ids.shape[-1]
    if uncond_embeddings is None:
        uncond_input = model.tokenizer(
            [""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt"
        )
        uncond_embeddings_ = model.text_encoder(uncond_input.input_ids.to(model.device))[0]
    else:
        uncond_embeddings_ = None

    latent, latents = ptp_utils.init_latent(latent, model, height, width, generator, batch_size)
    model.scheduler.set_timesteps(num_inference_steps)
    for i, t in enumerate(tqdm(model.scheduler.timesteps[-start_time:])):
        if uncond_embeddings_ is None:
            context = torch.cat([uncond_embeddings[i].expand(*text_embeddings.shape), text_embeddings])
        else:
            context = torch.cat([uncond_embeddings_, text_embeddings])
        latents = ptp_utils.diffusion_step(model, controller, latents, context, t, guidance_scale, low_resource=False)
        
    if return_type == 'image':
        image = ptp_utils.latent2image(model.vae, latents)
    else:
        image = latents
    return image, latent



def run_and_display(prompts, controller, latent=None, run_baseline=False, generator=None, uncond_embeddings=None, verbose=True):
    if run_baseline:
        print("w.o. prompt-to-prompt")
        images, latent = run_and_display(prompts, EmptyControl(), latent=latent, run_baseline=False, generator=generator)
        print("with prompt-to-prompt")
    images, x_t = text2image_ldm_stable(ldm_stable, prompts, controller, latent=latent, num_inference_steps=NUM_DIFFUSION_STEPS, guidance_scale=GUIDANCE_SCALE, generator=generator, uncond_embeddings=uncond_embeddings)
    if verbose:
        ptp_utils.view_images(images)
    return images, x_t

In [ ]:
import matplotlib.pyplot as plt
def save_plot(image_data, filename):
    plt.figure()
    plt.imshow(image_data)
    plt.axis('off')  # 不显示坐标轴
    plt.savefig(filename, bbox_inches='tight', pad_inches=0)
    plt.close()

In [ ]:
image_path = "./example_images/sculpture_29.jpg"
prompt = "A sculpture of a panda"
(image_gt, image_enc), x_t, uncond_embeddings = null_inversion.invert(image_path, prompt, verbose=True)

print("Modify or remove offsets according to your image!")

In [ ]:
prompts = [prompt]
controller = AttentionStore()
image_inv, x_t = run_and_display(prompts, controller, run_baseline=False, latent=x_t, uncond_embeddings=uncond_embeddings, verbose=False)
print("showing from left to right: the ground truth image, the vq-autoencoder reconstruction, the null-text inverted image")
ptp_utils.view_images([image_gt, image_enc, image_inv[0]])
show_cross_attention(controller, 16, ["up", "down"])

In [ ]:
prompts = ["A sculpture of a panda",
           "A sculpture of a bear"
        ]

cross_replace_steps = {'default_': .8,}
self_replace_steps = .5
blend_word = ((('panda',), ("bear",))) # for local edit. If it is not local yet - use only the source object: blend_word = ((('cat',), ("cat",))).
eq_params = {"words": ("bear",), "values": (2,)} # amplify attention to the word "tiger" by *2 

controller = make_controller(prompts, True, cross_replace_steps, self_replace_steps, blend_word, eq_params)
images, _ = run_and_display(prompts, controller, run_baseline=False, latent=x_t, uncond_embeddings=uncond_embeddings)
image = images[1]
save_plot(image, 'images/imagenul_7_1.png')

print("Image is highly affected by the self_replace_steps, usually 0.4 is a good default value, but you may want to try the range 0.3,0.4,0.5,0.7 ")

In [ ]:
prompts = ["A photo of a horse",
           "A photo of a robot horse"
        ]

cross_replace_steps = {'default_': .8, }
self_replace_steps = .6
blend_word = ((('horse',), ("horse",))) # for local edit
eq_params = {"words": ("robot",  ), "values": (2,)}  # amplify attention to the words "silver" and "sculpture" by *2 
 
controller = make_controller(prompts, False, cross_replace_steps, self_replace_steps, blend_word, eq_params)
images, _ = run_and_display(prompts, controller, run_baseline=False, latent=x_t, uncond_embeddings=uncond_embeddings)
image = images[1]
save_plot(image, 'images/imageptp_3_1.png')

In [ ]:
prompts = ["a cat sitting next to a mirror",
           "watercolor painting of a cat sitting next to a mirror"
        ]

cross_replace_steps = {'default_': .8, }
self_replace_steps = .7
blend_word = None
eq_params = {"words": ("watercolor",  ), "values": (5, 2,)}  # amplify attention to the word "watercolor" by 5
 
controller = make_controller(prompts, False, cross_replace_steps, self_replace_steps, blend_word, eq_params)
images, _ = run_and_display(prompts, controller, run_baseline=False, latent=x_t, uncond_embeddings=uncond_embeddings)

## Reconstruct Evaluation

In [6]:
import os, gc
import lpips
import pandas as pd
from datasets import load_dataset
from skimage.metrics import structural_similarity as SSIM
from skimage.metrics import peak_signal_noise_ratio as PSNR

def calculate_metrics(original_image, reconstructed_image, LPIPS=lpips.LPIPS(net='vgg').to(device)):
    """
    计算 LPIPS, PSNR, 和 SSIM 指标
    """
    # 转换为 PyTorch Tensor 并归一化到 [-1, 1]
    def to_tensor(img):
        img_t = torch.tensor(img).float().permute(2, 0, 1) / 127.5 - 1
        return img_t.unsqueeze(0).to(device)

    original_tensor = to_tensor(original_image)
    reconstructed_tensor = to_tensor(reconstructed_image)

    # 计算 LPIPS
    lpips_value = LPIPS(original_tensor, reconstructed_tensor).item()

    # 计算 PSNR
    psnr_value = PSNR(original_image, reconstructed_image)

    # 计算 SSIM
    ssim_value = SSIM(original_image, reconstructed_image, channel_axis=-1, data_range=255) # 注意 channel_axis 和 data_range

    return lpips_value, psnr_value, ssim_value

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/djm/ERDDCI/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/djm/ERDDCI/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/djm/ERDDCI/.venv/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth


In [7]:
image_folder = "./ReconDataset/DOCCI/" # wild-ti2i, imagenetr-ti2i or DOCCI 
output_folder = os.path.join(image_folder, f"recon_G{int(GUIDANCE_SCALE)}S{NUM_DIFFUSION_STEPS}")
results_file = os.path.join(output_folder, f"nti_results_G{int(GUIDANCE_SCALE)}S{NUM_DIFFUSION_STEPS}.txt")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Creating output folder: {output_folder}")

if "DOCCI" in image_folder:
    recon_test_dataset = load_dataset("google/docci", split="test")
else:
    recon_test_dataset = load_dataset(image_folder, split="test", trust_remote_code=True)

print(recon_test_dataset)
print(recon_test_dataset.info.dataset_name)
print(recon_test_dataset[0])


Using the latest cached version of the module from /home/djm/.cache/huggingface/modules/datasets_modules/datasets/google--docci/d38d507c9795602d4e50dcd02e1e1dc4fa8c58ac69c2949c5c3a20c9d00c5b8b (last modified on Mon Apr 21 14:23:53 2025) since it couldn't be found locally at google/docci, or remotely on the Hugging Face Hub.


Dataset({
    features: ['image', 'example_id', 'description'],
    num_rows: 5000
})
docci
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=2048x1536 at 0x7F02780BB1D0>, 'example_id': 'test_00000', 'description': 'A high angle view of an old faded street corner. In the middle of the view is the orange spray painted word "ROW", with a horizontal letter "i" placed above it. On the right side of the image is a partially visible and faded red line on the street corner with the words " FIRE LANE", heavily faded in white paint.'}


In [ ]:
results_data = []
RECONSTRUCT_NUM_LIMIT = 500
loop_counter = 0
with open(results_file, 'w') as f:
    f.write(f"Reconstruction Test Results (Input Folder: {image_folder}, Output Folder: {output_folder}):\n")

    for item in recon_test_dataset:
        if loop_counter >= RECONSTRUCT_NUM_LIMIT:
            break
        image_or_path = item['image']
        prompt = item['description']
        if recon_test_dataset.info.dataset_name == 'docci':
            filename = f"{item['example_id']}.png"
        else:
            filename = os.path.basename(image_or_path)
        
        print(f"Processing: {filename}, {prompt=}")
        f.write(f"Processing image: {filename}\n")

        # Load image
        original_image = load_512(image_or_path)
        if original_image is None:
            continue

        try:
            # invert and inference
            encode_latent, ddim_invert_latent, nti_inference_latent = \
                    null_inversion.get_encode_invert_inference_latent(image_or_path, prompt, guidance_scale=GUIDANCE_SCALE)
            
            # Decode
            recon_image_encode = null_inversion.latent2image(encode_latent)
            recon_image_nti = null_inversion.latent2image(nti_inference_latent)

            # Calculate evaluation metrics
            metrics_nti = calculate_metrics(recon_image_encode, recon_image_nti)

            # Save result
            results_data.append({
                'filename': filename,
                'nti_lpips': metrics_nti[0],
                'nti_psnr': metrics_nti[1],
                'nti_ssim': metrics_nti[2],
            })
            print(f"{filename} - NTI: LPIPS={metrics_nti[0]:.4f}, PSNR={metrics_nti[1]:.4f}, SSIM={metrics_nti[2]:.4f}")

            # Save recon images
            # output_image_encode = Image.fromarray(recon_image_encode)
            # output_path_encode = os.path.join(output_folder, f"recon_encode_{filename}")
            # output_image_encode.save(output_path_encode)
            # print(f"  Reconstructed (encode) saved to: {output_path_encode}")

            output_image_nti = Image.fromarray(recon_image_nti)
            output_path_nti = os.path.join(output_folder, f"recon_nti_{filename}")
            output_image_nti.save(output_path_nti)
            print(f"  Reconstructed (NTI) saved to: {output_path_nti}")

            loop_counter += 1
        except Exception as e:
            print(f"  Error processing {filename}: {e}")
            break

    df_results = pd.DataFrame(results_data)
    average_metrics = df_results[['nti_lpips', 'nti_psnr', 'nti_ssim']].mean()
    print("\nAverage Metrics:")
    print(average_metrics)

    # 将 DataFrame 保存为 CSV 文件
    csv_output_file = results_file.replace('.txt', '.csv')
    df_results.to_csv(csv_output_file, index=False)
    print(f"\nResults saved to {csv_output_file}")

    json_output_file = results_file.replace('.txt', '.json')
    df_results.to_json(json_output_file, orient='records', indent=4)
    print(f"Results saved to {json_output_file}")
    
print("Reconstruction test completed! Results are saved in reconstruction_results.txt, and reconstructed images are in the corresponding '_recon' folder.")

Processing: test_00000.png, prompt='A high angle view of an old faded street corner. In the middle of the view is the orange spray painted word "ROW", with a horizontal letter "i" placed above it. On the right side of the image is a partially visible and faded red line on the street corner with the words " FIRE LANE", heavily faded in white paint.'


  0%|          | 0/100 [00:00<?, ?it/s]

test_00000.png - NTI: LPIPS=0.2499, PSNR=24.4847, SSIM=0.5788
  Reconstructed (NTI) saved to: ./ReconDataset/DOCCI/recon_G3S10/recon_nti_test_00000.png
Processing: test_00001.png, prompt='An outdoor front view of a turtle that is sitting on a floating tree trunk that has moss growing at the front of it. The turtle is yellow and green and has a dark green shell. The turtle is pointing his head up and soaking up the sun. On the water, there are a couple pieces of foam floating in the swamp. In the far background, there are multiple dried pieces of grass. On the far left side of the swamp, there is a fallen tree trunk that has moss on it.'


  0%|          | 0/100 [00:00<?, ?it/s]

test_00001.png - NTI: LPIPS=0.1879, PSNR=22.2847, SSIM=0.7562
  Reconstructed (NTI) saved to: ./ReconDataset/DOCCI/recon_G3S10/recon_nti_test_00001.png
Processing: test_00002.png, prompt='An outdoor shot, looking up at the golden statue of a woman with three mythical seahorses sitting atop a gray brick monument. The woman is facing forward with her left arm raised up, holding a small leafy branch in her hand. She is holding a long cylinder vertically against her body with her right arm and has a large circular shield on her back. She has a leaf crown on her head with long braids that are visible flowing on the left side. The three mythical horses are spread evenly in front of her feet. The horse to the left has its mouth wide open. The horse in the middle has its chin resting on its chest. The horse on the right is a side view with the head facing the right and the mouth open. The legs are depicted as if they are moving. The background is a clear blue sky. Daytime.'


...

  0%|          | 0/100 [00:00<?, ?it/s]

test_00024.png - NTI: LPIPS=0.1091, PSNR=22.3799, SSIM=0.8359
  Reconstructed (NTI) saved to: ./ReconDataset/DOCCI/recon_G3S10/recon_nti_test_00024.png
Processing: test_00025.png, prompt='An outdoors top-down view of purple sidewalk chalk on a concrete sidewalk reading,"Fear / of / Chores". The left side of the concrete slab has green algae growth that fades to the right. Small bits of smashed acorns are scattered across the slab.'


  0%|          | 0/100 [00:00<?, ?it/s]

test_00025.png - NTI: LPIPS=0.2902, PSNR=23.1636, SSIM=0.6996
  Reconstructed (NTI) saved to: ./ReconDataset/DOCCI/recon_G3S10/recon_nti_test_00025.png
Processing: test_00026.png, prompt="An indoor close up shot of a small, plastic red toy car floating in a white bathtub. The toy car has black windows, bumpers, and tires. The car is positioned diagonally, with the front bumper pointed towards the lower left of the image. Small bits of white light reflect on the water's surface, and there are a few small bubbles scattered on the surface of the water."


  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
image_folder = "./ReconDataset/wild-ti2i/" # wild-ti2i, imagenetr-ti2i or DOCCI 
output_folder = os.path.join(image_folder, f"recon_G{int(GUIDANCE_SCALE)}S{NUM_DIFFUSION_STEPS}")
results_file = os.path.join(output_folder, f"nti_results_G{int(GUIDANCE_SCALE)}S{NUM_DIFFUSION_STEPS}.txt")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Creating output folder: {output_folder}")

if "DOCCI" in image_folder:
    recon_test_dataset = load_dataset("google/docci", split="test")
else:
    recon_test_dataset = load_dataset(image_folder, split="test", trust_remote_code=True)

print(recon_test_dataset)
print(recon_test_dataset.info.dataset_name)
print(recon_test_dataset[0])

In [ ]:
results_data = []
RECONSTRUCT_NUM_LIMIT = 500
loop_counter = 0
with open(results_file, 'w') as f:
    f.write(f"Reconstruction Test Results (Input Folder: {image_folder}, Output Folder: {output_folder}):\n")

    for item in recon_test_dataset:
        if loop_counter >= RECONSTRUCT_NUM_LIMIT:
            break
        image_or_path = item['image']
        prompt = item['description']
        if recon_test_dataset.info.dataset_name == 'docci':
            filename = f"{item['example_id']}.png"
        else:
            filename = os.path.basename(image_or_path)
        
        print(f"Processing: {filename}, {prompt=}")
        f.write(f"Processing image: {filename}\n")

        # Load image
        original_image = load_512(image_or_path)
        if original_image is None:
            continue

        try:
            # invert and inference
            encode_latent, ddim_invert_latent, nti_inference_latent = \
                    null_inversion.get_encode_invert_inference_latent(image_or_path, prompt, guidance_scale=GUIDANCE_SCALE)
            
            # Decode
            recon_image_encode = null_inversion.latent2image(encode_latent)
            recon_image_nti = null_inversion.latent2image(nti_inference_latent)

            # Calculate evaluation metrics
            metrics_nti = calculate_metrics(recon_image_encode, recon_image_nti)

            # Save result
            results_data.append({
                'filename': filename,
                'nti_lpips': metrics_nti[0],
                'nti_psnr': metrics_nti[1],
                'nti_ssim': metrics_nti[2],
            })
            print(f"{filename} - NTI: LPIPS={metrics_nti[0]:.4f}, PSNR={metrics_nti[1]:.4f}, SSIM={metrics_nti[2]:.4f}")

            # Save recon images
            # output_image_encode = Image.fromarray(recon_image_encode)
            # output_path_encode = os.path.join(output_folder, f"recon_encode_{filename}")
            # output_image_encode.save(output_path_encode)
            # print(f"  Reconstructed (encode) saved to: {output_path_encode}")

            output_image_nti = Image.fromarray(recon_image_nti)
            output_path_nti = os.path.join(output_folder, f"recon_nti_{filename}")
            output_image_nti.save(output_path_nti)
            print(f"  Reconstructed (NTI) saved to: {output_path_nti}")

            loop_counter += 1
        except Exception as e:
            print(f"  Error processing {filename}: {e}")
            break

    df_results = pd.DataFrame(results_data)
    average_metrics = df_results[['nti_lpips', 'nti_psnr', 'nti_ssim']].mean()
    print("\nAverage Metrics:")
    print(average_metrics)

    # 将 DataFrame 保存为 CSV 文件
    csv_output_file = results_file.replace('.txt', '.csv')
    df_results.to_csv(csv_output_file, index=False)
    print(f"\nResults saved to {csv_output_file}")

    json_output_file = results_file.replace('.txt', '.json')
    df_results.to_json(json_output_file, orient='records', indent=4)
    print(f"Results saved to {json_output_file}")
    
print("Reconstruction test completed! Results are saved in reconstruction_results.txt, and reconstructed images are in the corresponding '_recon' folder.")

In [ ]:
image_folder = "./ReconDataset/imagenetr-ti2i/" # wild-ti2i, imagenetr-ti2i or DOCCI 
output_folder = os.path.join(image_folder, f"recon_G{int(GUIDANCE_SCALE)}S{NUM_DIFFUSION_STEPS}")
results_file = os.path.join(output_folder, f"nti_results_G{int(GUIDANCE_SCALE)}S{NUM_DIFFUSION_STEPS}.txt")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Creating output folder: {output_folder}")

if "DOCCI" in image_folder:
    recon_test_dataset = load_dataset("google/docci", split="test")
else:
    recon_test_dataset = load_dataset(image_folder, split="test", trust_remote_code=True)

print(recon_test_dataset)
print(recon_test_dataset.info.dataset_name)
print(recon_test_dataset[0])

In [ ]:
results_data = []
RECONSTRUCT_NUM_LIMIT = 500
loop_counter = 0
with open(results_file, 'w') as f:
    f.write(f"Reconstruction Test Results (Input Folder: {image_folder}, Output Folder: {output_folder}):\n")

    for item in recon_test_dataset:
        if loop_counter >= RECONSTRUCT_NUM_LIMIT:
            break
        image_or_path = item['image']
        prompt = item['description']
        if recon_test_dataset.info.dataset_name == 'docci':
            filename = f"{item['example_id']}.png"
        else:
            filename = os.path.basename(image_or_path)
        
        print(f"Processing: {filename}, {prompt=}")
        f.write(f"Processing image: {filename}\n")

        # Load image
        original_image = load_512(image_or_path)
        if original_image is None:
            continue

        try:
            # invert and inference
            encode_latent, ddim_invert_latent, nti_inference_latent = \
                    null_inversion.get_encode_invert_inference_latent(image_or_path, prompt, guidance_scale=GUIDANCE_SCALE)
            
            # Decode
            recon_image_encode = null_inversion.latent2image(encode_latent)
            recon_image_nti = null_inversion.latent2image(nti_inference_latent)

            # Calculate evaluation metrics
            metrics_nti = calculate_metrics(recon_image_encode, recon_image_nti)

            # Save result
            results_data.append({
                'filename': filename,
                'nti_lpips': metrics_nti[0],
                'nti_psnr': metrics_nti[1],
                'nti_ssim': metrics_nti[2],
            })
            print(f"{filename} - NTI: LPIPS={metrics_nti[0]:.4f}, PSNR={metrics_nti[1]:.4f}, SSIM={metrics_nti[2]:.4f}")

            # Save recon images
            # output_image_encode = Image.fromarray(recon_image_encode)
            # output_path_encode = os.path.join(output_folder, f"recon_encode_{filename}")
            # output_image_encode.save(output_path_encode)
            # print(f"  Reconstructed (encode) saved to: {output_path_encode}")

            output_image_nti = Image.fromarray(recon_image_nti)
            output_path_nti = os.path.join(output_folder, f"recon_nti_{filename}")
            output_image_nti.save(output_path_nti)
            print(f"  Reconstructed (NTI) saved to: {output_path_nti}")

            loop_counter += 1
        except Exception as e:
            print(f"  Error processing {filename}: {e}")
            break

    df_results = pd.DataFrame(results_data)
    average_metrics = df_results[['nti_lpips', 'nti_psnr', 'nti_ssim']].mean()
    print("\nAverage Metrics:")
    print(average_metrics)

    # 将 DataFrame 保存为 CSV 文件
    csv_output_file = results_file.replace('.txt', '.csv')
    df_results.to_csv(csv_output_file, index=False)
    print(f"\nResults saved to {csv_output_file}")

    json_output_file = results_file.replace('.txt', '.json')
    df_results.to_json(json_output_file, orient='records', indent=4)
    print(f"Results saved to {json_output_file}")
    
print("Reconstruction test completed! Results are saved in reconstruction_results.txt, and reconstructed images are in the corresponding '_recon' folder.")